## Selection of k with the Prediction Strength

This notebook selects the number of clusters k for the K-Means analysis with the **prediction strength** of Tibshirani & Walther (2005, *Cluster Validation by Prediction Strength*, Journal of Computational and Graphical Statistics). 

The idea is that a number of clusters is only meaningful if the same clusters can be found again in new data.

- **Reference run:** the average hidden states of the evenly spaced tasks from `kmeans_evenly_spaced.ipynb`. K-Means is fitted on them.
- **Repeated rollouts:** the same evenly spaced tasks, played `num_repeats` more times with different seeds. Every repetition is clustered on its own with K-Means.
- **Prediction strength of one repetition:** the repetition is clustered on its own. For each of its clusters, every pair of tasks in it is checked: are both tasks also assigned to the same reference centroid? The share of such pairs is calculated per cluster, and the weakest cluster determines the value. So a single cluster that cannot be found again lowers the prediction strength.
- **Selection:** the prediction strength is averaged over all repetitions. The selected k is the largest k with a mean prediction strength above `cutoff` = 0.8. If no k reaches the threshold, there is no reproducible cluster structure, and k = 1 is reported.

In [1]:
import numpy as np
import os
import sys

sys.path.insert(0, os.path.abspath("../.."))
sys.path.insert(0, os.path.abspath(".."))
from analysis_utils import SYSTEMS, CHECKPOINT_DIR, set_seeds, load_model, collect_avg_hidden_states, task_parameters
from kmeans_utils import fit_kmeans, prediction_strength

system = SYSTEMS["halfcheetah"]
num_repeats = 20
steps_per_env = 3000

k_values = np.arange(2, 11)
cutoff = 0.8

repeat_seeds = 2000 + np.arange(num_repeats)
set_seeds(42)

# Reference run
avg_hidden_states = np.load(
    CHECKPOINT_DIR / "halfcheetah_action_avg_hidden_states.npy"
)
num_envs = len(avg_hidden_states)

env_configs = system.sample_tasks_eval(num_envs=num_envs)
offset_values = task_parameters(system, env_configs)

print(avg_hidden_states.shape)

(20, 256)


In [2]:
# Create the RL2 model and load the trained weights
model = load_model(system)

### Repeated Rollouts of the Evenly Spaced Tasks

The evenly spaced tasks are played `num_repeats` times for `steps_per_env` steps.

In [3]:
# Average hidden state of every task in every repetition
repeat_avg_hidden_states = np.zeros(
    (num_repeats, num_envs, model.feat_extractor.h_dim),
    dtype=avg_hidden_states.dtype
)

envs = system.make_envs(env_configs)

try:
    for repeat, seed in enumerate(repeat_seeds):
        repeat_avg_hidden_states[repeat] = collect_avg_hidden_states(
            model, system, envs, seed=int(seed), steps=steps_per_env
        )
        print(f"Repetition {repeat + 1}/{num_repeats} completed")
finally:
    envs.close()

Repetition 1/20 completed
Repetition 2/20 completed
Repetition 3/20 completed
Repetition 4/20 completed
Repetition 5/20 completed
Repetition 6/20 completed
Repetition 7/20 completed
Repetition 8/20 completed
Repetition 9/20 completed
Repetition 10/20 completed
Repetition 11/20 completed
Repetition 12/20 completed
Repetition 13/20 completed
Repetition 14/20 completed
Repetition 15/20 completed
Repetition 16/20 completed
Repetition 17/20 completed
Repetition 18/20 completed
Repetition 19/20 completed
Repetition 20/20 completed


### Prediction Strength per k

1. K-Means is fitted on the hidden states of the reference run.
2. Every repetition is clustered on its own with K-Means.
3. For every cluster of the repetition the share of task pairs that also share a reference centroid is calculated. The weakest cluster gives the prediction strength of this repetition.

In [4]:
# Prediction strength of every repetition for every k
prediction_strengths = np.zeros((num_repeats, len(k_values)))

for j, k in enumerate(k_values):
    reference_kmeans = fit_kmeans(avg_hidden_states, k)

    for repeat in range(num_repeats):
        prediction_strengths[repeat, j] = prediction_strength(
            reference_kmeans, repeat_avg_hidden_states[repeat], k
        )

mean_prediction_strength = prediction_strengths.mean(axis=0)
std_prediction_strength = prediction_strengths.std(axis=0, ddof=1)

# Largest k whose mean prediction strength is above the threshold
passing_k = k_values[mean_prediction_strength > cutoff]

# If no k reaches the threshold, there is no reproducible cluster structure (k = 1)
best_k = int(passing_k.max()) if len(passing_k) > 0 else 1

print(f"{'k':>3} | {'Mean PS':>7} | {'Std':>5} | {'Min':>5}")

for j, k in enumerate(k_values):
    marker = "  <- selected" if k == best_k else ""
    print(
        f"{k:>3} | {mean_prediction_strength[j]:>7.3f} | {std_prediction_strength[j]:>5.3f} | "
        f"{prediction_strengths[:, j].min():>5.3f}{marker}"
    )

if best_k > 1:
    print(f"\nSelected k = {best_k} (largest k with a mean prediction strength above {cutoff})")
else:
    print(f"\nNo k reaches a mean prediction strength above {cutoff}: no reproducible cluster structure (k = 1)")

  k | Mean PS |   Std |   Min
  2 |   0.781 | 0.219 | 0.471
  3 |   0.710 | 0.235 | 0.314
  4 |   0.496 | 0.142 | 0.400
  5 |   0.368 | 0.119 | 0.000
  6 |   0.277 | 0.220 | 0.000
  7 |   0.237 | 0.246 | 0.000
  8 |   0.183 | 0.247 | 0.000
  9 |   0.183 | 0.259 | 0.000
 10 |   0.092 | 0.239 | 0.000

No k reaches a mean prediction strength above 0.8: no reproducible cluster structure (k = 1)


In [5]:
# Save the results and corresponding rollout data
np.savez(
    CHECKPOINT_DIR / "halfcheetah_action_kmeans_k_selection.npz",
    k_values=k_values,
    cutoff=cutoff,
    best_k=best_k,
    prediction_strengths=prediction_strengths,
    mean_prediction_strength=mean_prediction_strength,
    offset_values=offset_values,
    repeat_seeds=repeat_seeds,
    repeat_avg_hidden_states=repeat_avg_hidden_states
)

print("k selection results saved.")

k selection results saved.
